In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from pathlib import Path
import pynapple as nap

from scipy.stats import gaussian_kde

from spatial_manifolds.data.binning import get_bin_config
from spatial_manifolds.data.loading import load_session
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2


/Users/harryclark/Documents/spatial-manifolds/src/spatial_manifolds/detect_grids.py:609: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if vr_type is 'MCVR':
/Users/harryclark/Documents/spatial-manifolds/src/spatial_manifolds/detect_grids.py:1091: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if vr_type is 'MCVR':


: 

In [ ]:
# Load session
mouse_days = {
              25: [26,27,28,29,30,31],
              26: [20,21,22,23,24],
              27: [25,27,28,29,30],
              28: [24,26,27,28,29],
              29: [24,26,27,28,29],
            }

for mouse, days in mouse_days.items():
    for day in days:

        tcs, tcs_time, autocorrs, last_ephys_bin, beh, clusters_VR = compute_vr_tcs(mouse, day, vr_type='MCVR',
                                                                                    source_path='/Users/harryclark/Downloads/COHORT12_OLD/')
        cluster_ids = clusters_VR.index
        plot_individual_rate_maps_with_avg_by_trial_type(mouse, day, cluster_ids, label='',
                                                         figpath='/Users/harryclark/Desktop/mcvr_rate_maps/', vr_type='MCVR',
                                                         source_path='/Users/harryclark/Downloads/COHORT12_OLD/')
        plt.close('all')

In [ ]:
brain_coord_SC = np.array([2500, 3500, 4200]) # ec
brain_coord_SC = np.array([2500, 4200, 3500]) # CA1

brain_coord_SC = np.array([3500, 2500, 4200]) # TEa5
brain_coord_SC = np.array([3500, 4200, 2500]) # root

brain_coord_SC = np.array([4200, 2500, 3500]) # ENTm5 <- this is the right way around
#brain_coord_SC = np.array([4200, 3500, 2500]) # root

# coords go AP, DV, ML
ap_axis = np.arange(3500, 5000, 10)
dv_axis = np.arange(1000, 5000, 10)
ml_axis = np.arange(2500, 4000, 10)

annotations = np.zeros((len(ap_axis), len(dv_axis), len(ml_axis)), dtype=object)
for z, ap in enumerate(ap_axis):
    print(f'Processing AP coordinate: {ap}/{ap_axis[-1]}')
    for y, dv in enumerate(dv_axis):
        for x, ml in enumerate(ml_axis):
            brain_coord_SC = np.array([ap, dv, ml])
            brain_coord_CCF = StereoToCCF(brain_coord_SC)
            z_CCF, y_CCF, x_CCF = np.round(brain_coord_CCF / 10).astype(int)
            annotation_index = annotations_set[z_CCF, y_CCF, x_CCF]

            if len(structure_set[structure_set['id'] == annotation_index]) ==1:
                annotation = structure_set[structure_set['id'] == annotation_index]['acronym'].iloc[0]
                annotations[z, y, x] = annotation
            else:
                annotations[z,y,x] = 'root'


In [ ]:
# set colors based on annotations
annotation_colors = np.zeros((len(ap_axis), len(dv_axis), len(ml_axis)), dtype=object)
# Set colors to black where annotation contains 'VIS'
for z in range(annotation_colors.shape[0]):
    for y in range(annotation_colors.shape[1]):
        for x in range(annotation_colors.shape[2]):
            if 'VIS' in str(annotations[z, y, x]):
                annotation_colors[z, y, x] = 'black'
            elif 'ENT' in str(annotations[z, y, x]):
                annotation_colors[z, y, x] = 'silver'
            elif 'RSP' in str(annotations[z, y, x]):
                annotation_colors[z, y, x] = 'grey'
            elif 'SUB' in str(annotations[z, y, x]):
                annotation_colors[z, y, x] = 'gainsboro'
            elif 'PAR' in str(annotations[z, y, x]):
                annotation_colors[z, y, x] = 'dimgrey'
            elif 'PRE' in str(annotations[z, y, x]):
                annotation_colors[z, y, x] = 'dimgrey'
            elif 'POST' in str(annotations[z, y, x]):
                annotation_colors[z, y, x] = 'lightslategrey'
            elif 'HPF' in str(annotations[z, y, x]):
                annotation_colors[z, y, x] = 'lightslategrey'
            else:
                annotation_colors[z, y, x] = 'white'


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import imageio
from matplotlib.patches import Patch

frames = []
for i in range(annotations.shape[0]):
    ap_annotations = annotations[i, :, :]
    ap_colors_i = annotation_colors[i, :, :]

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    unique_colors = np.unique(ap_colors_i)
    for color in unique_colors:
        border_points = extract_border(ap_colors_i, color, only_border=False)
        if color != 'red':
            ml_points = ml_axis[border_points[:, 1]]
            dv_points = dv_axis[border_points[:, 0]]
            ax.scatter(ml_points, dv_points, color=color, s=1, label=color)
    ax.invert_yaxis()
    ax.set_title(f'AP: {ap_axis[i]}')
    plt.tight_layout()

    # Save frame to a buffer (Jupyter compatible)
    fig.canvas.draw()
    image = np.array(fig.canvas.buffer_rgba())
    frames.append(image)
    plt.close(fig)

# Save as GIF
gif_path = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration/brain_regions_across_ap.gif'
imageio.mimsave(gif_path, frames, duration=0.3)
print(f"GIF saved to {gif_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# Prepare data
ml_ns = ns_['coord_SCs_x'] * -1
ml_ngs = ngs_['coord_SCs_x'] * -1
ml_gcs = gcs_['coord_SCs_x'] * -1

dv_ns = ns_['coord_SCs_y']
dv_ngs = ngs_['coord_SCs_y']
dv_gcs = gcs_['coord_SCs_y']

ap_ns = ns_['coord_SCs_z']
ap_ngs = ngs_['coord_SCs_z']
ap_gcs = gcs_['coord_SCs_z']

# AP slices for annotation overlay (bottom left plot only)
ap_slices = [4000, 4100, 4200, 4300, 4400]
ap_indices = [np.where(ap_axis == ap)[0][0] for ap in ap_slices]

# ML slices for annotation overlay (bottom middle plot only)
ml_slices = [2800, 3100, 3400, 3700]
ml_indices = [np.where(ml_axis == ml)[0][0] for ml in ml_slices]

# Color legend for annotation
legend_labels = {
    'silver': 'ENT',
    'black': 'VIS',
    'grey': 'RSP',
    'cyan': 'SUB',
    'dimgrey': 'PAR/PRE',
    'yellow': 'HPF',
    'black': 'root/other'
}

fig, axes = plt.subplots(2, 3, figsize=(5, 4), width_ratios=[1, 0.5, 0.2], height_ratios=[0.2, 1], gridspec_kw={'wspace': 0.15, 'hspace': 0.15}, sharex='col', sharey='row')

# Top left: ML histogram
axes[0, 0].hist(ml_ngs, bins=10, color='#CAC841', alpha=0.7, label='ngs', density=True)
axes[0, 0].hist(ml_gcs, bins=10, color='#6897CD', alpha=0.7, label='gcs', density=True)
axes[0, 0].axis('off')

# Top middle: AP histogram
axes[0, 1].hist(ap_ngs, bins=5, color='#CAC841', alpha=0.7, label='ngs', density=True)
axes[0, 1].hist(ap_gcs, bins=5, color='#6897CD', alpha=0.7, label='gcs', density=True)
axes[0, 1].axis('off')

# Top right: DV histogram (horizontal)
axes[1, 2].hist(dv_ngs, bins=10, color='#CAC841', alpha=0.7, orientation='horizontal', label='ngs', density=True)
axes[1, 2].hist(dv_gcs, bins=10, color="#6897CD", alpha=0.7, orientation='horizontal', label='gcs', density=True)
axes[1, 2].axis('off')

# Bottom left: ML vs DV scatter with annotation overlays for AP=4200, 4300, 4400
ax_ml_dv = axes[1, 0]
for ap_idx, ap_val in zip(ap_indices, ap_slices):
    ap_colors_i = annotation_colors[ap_idx, :, :]
    ap_annotations_i = annotations[ap_idx, :, :]
    #print(f'unique annotations at AP={ap_val}: {np.unique(ap_annotations_i)}')
    unique_colors = np.unique(ap_colors_i)
    for color in unique_colors:
        points = extract_border(ap_colors_i, color, only_border=False)
        ml_points = ml_axis[points[:, 1]]
        dv_points = dv_axis[points[:, 0]]
        ax_ml_dv.scatter(ml_points, dv_points, color=color, s=1, alpha=0.05, zorder=1, rasterized=True)
ax_ml_dv.scatter(ml_ngs, dv_ngs, color='#CAC841', alpha=0.3, s=10, label='ngs', zorder=2)
ax_ml_dv.scatter(ml_gcs, dv_gcs, color='#6897CD', alpha=0.3, s=10, label='gcs', zorder=2)
ax_ml_dv.set_xlim(2500, 4000)
ax_ml_dv.set_ylim(1000, 5000)
ax_ml_dv.invert_yaxis()
ax_ml_dv.set_xlabel('Medial-lateral (µm)')
ax_ml_dv.set_ylabel('Dorsal-ventral (µm)')

# Bottom middle: AP vs DV scatter (no annotation overlay)
ax_ap_dv = axes[1, 1]
for ml_idx, ml_val in zip(ml_indices, ml_slices):
    ml_colors_i = annotation_colors[:, :, ml_idx]
    unique_colors = np.unique(ml_colors_i)
    for color in unique_colors:
        points = extract_border(ml_colors_i, color, only_border=False)
        ap_points = ap_axis[points[:, 0]]
        dv_points = dv_axis[points[:, 1]]
        ax_ap_dv.scatter(ap_points, dv_points, color=color, s=1, alpha=0.05, zorder=1, rasterized=True)
axes[1, 1].scatter(ap_ngs, dv_ngs, color='#CAC841', alpha=0.15, s=10, label='ngs')
axes[1, 1].scatter(ap_gcs, dv_gcs, color='#6897CD', alpha=0.15, s=10, label='gcs')
axes[1, 1].set_xlim(3500, 5000)
axes[1, 1].set_xlabel('Anterior-posterior (µm)')

# Bottom right: blank
axes[0, 2].axis('off')

plt.tight_layout()
plt.savefig('/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration/cluster_locations_joint_grid_with_annotations_overlay.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# List of unique mice in your data
all_mice = np.unique(gcs_['mouse'].tolist() + ngs_['mouse'].tolist())

for mouse in all_mice:
    # Filter for this mouse
    gcs_mouse = gcs_[gcs_['mouse'] == mouse]
    ngs_mouse = ngs_[ngs_['mouse'] == mouse]

    ml_gcs = gcs_mouse['coord_SCs_x'] * -1
    ml_ngs = ngs_mouse['coord_SCs_x'] * -1
    dv_gcs = gcs_mouse['coord_SCs_y']
    dv_ngs = ngs_mouse['coord_SCs_y']
    ap_gcs = gcs_mouse['coord_SCs_z']
    ap_ngs = ngs_mouse['coord_SCs_z']

    fig, axes = plt.subplots(2, 3, figsize=(5, 4), width_ratios=[1, 0.5, 0.2], height_ratios=[0.2, 1], gridspec_kw={'wspace': 0.15, 'hspace': 0.15}, sharex='col', sharey='row')

    # Top left: ML histogram
    axes[0, 0].hist(ml_ngs, bins=10, color='#CAC841', alpha=0.7, label='ngs', density=True)
    axes[0, 0].hist(ml_gcs, bins=10, color='#6897CD', alpha=0.7, label='gcs', density=True)
    axes[0, 0].axis('off')

    # Top middle: AP histogram
    axes[0, 1].hist(ap_ngs, bins=5, color='#CAC841', alpha=0.7, label='ngs', density=True)
    axes[0, 1].hist(ap_gcs, bins=5, color='#6897CD', alpha=0.7, label='gcs', density=True)
    axes[0, 1].axis('off')

    # Top right: DV histogram (horizontal)
    axes[1, 2].hist(dv_ngs, bins=10, color='#CAC841', alpha=0.7, orientation='horizontal', label='ngs', density=True)
    axes[1, 2].hist(dv_gcs, bins=10, color="#6897CD", alpha=0.7, orientation='horizontal', label='gcs', density=True)
    axes[1, 2].axis('off')

    # Bottom left: ML vs DV scatter with annotation overlays for AP slices
    ax_ml_dv = axes[1, 0]
    for ap_idx, ap_val in zip(ap_indices, ap_slices):
        ap_colors_i = annotation_colors[ap_idx, :, :]
        unique_colors = np.unique(ap_colors_i)
        for color in unique_colors:
            points = extract_border(ap_colors_i, color, only_border=False)
            ml_points = ml_axis[points[:, 1]]
            dv_points = dv_axis[points[:, 0]]
            ax_ml_dv.scatter(ml_points, dv_points, color=color, s=1, alpha=0.05, zorder=1, rasterized=True)
    ax_ml_dv.scatter(ml_ngs, dv_ngs, color='#CAC841', alpha=0.3, s=10, label='ngs', zorder=2)
    ax_ml_dv.scatter(ml_gcs, dv_gcs, color='#6897CD', alpha=0.3, s=10, label='gcs', zorder=2)
    ax_ml_dv.set_xlim(2500, 4000)
    ax_ml_dv.set_ylim(1000, 5000)
    ax_ml_dv.invert_yaxis()
    ax_ml_dv.set_xlabel('Medial-lateral (µm)')
    ax_ml_dv.set_ylabel('Dorsal-ventral (µm)')

    # Bottom middle: AP vs DV scatter (no annotation overlay)
    ax_ap_dv = axes[1, 1]
    for ml_idx, ml_val in zip(ml_indices, ml_slices):
        ml_colors_i = annotation_colors[:, :, ml_idx]
        unique_colors = np.unique(ml_colors_i)
        for color in unique_colors:
            points = extract_border(ml_colors_i, color, only_border=False)
            ap_points = ap_axis[points[:, 0]]
            dv_points = dv_axis[points[:, 1]]
            ax_ap_dv.scatter(ap_points, dv_points, color=color, s=1, alpha=0.05, zorder=1, rasterized=True)
    ax_ap_dv.scatter(ap_ngs, dv_ngs, color='#CAC841', alpha=0.15, s=10, label='ngs')
    ax_ap_dv.scatter(ap_gcs, dv_gcs, color='#6897CD', alpha=0.15, s=10, label='gcs')
    ax_ap_dv.set_xlim(3500, 5000)
    ax_ap_dv.set_xlabel('Anterior-posterior (µm)')

    # Bottom right: blank
    axes[0, 2].axis('off')

    plt.suptitle(f'Mouse {mouse}', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration/cluster_locations_joint_grid_mouse_{mouse}.pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# Prepare data
theta_cut = 0.07

# For NGS
ml_ngs_theta = ngs_[ngs_['theta_index'] >= theta_cut]['coord_SCs_x'] * -1
ml_ngs_nontheta = ngs_[ngs_['theta_index'] < theta_cut]['coord_SCs_x'] * -1
dv_ngs_theta = ngs_[ngs_['theta_index'] >= theta_cut]['coord_SCs_y']
dv_ngs_nontheta = ngs_[ngs_['theta_index'] < theta_cut]['coord_SCs_y']
ap_ngs_theta = ngs_[ngs_['theta_index'] >= theta_cut]['coord_SCs_z']
ap_ngs_nontheta = ngs_[ngs_['theta_index'] < theta_cut]['coord_SCs_z']

# For GCS
ml_gcs_theta = gcs_[gcs_['theta_index'] >= theta_cut]['coord_SCs_x'] * -1
ml_gcs_nontheta = gcs_[gcs_['theta_index'] < theta_cut]['coord_SCs_x'] * -1
dv_gcs_theta = gcs_[gcs_['theta_index'] >= theta_cut]['coord_SCs_y']
dv_gcs_nontheta = gcs_[gcs_['theta_index'] < theta_cut]['coord_SCs_y']
ap_gcs_theta = gcs_[gcs_['theta_index'] >= theta_cut]['coord_SCs_z']
ap_gcs_nontheta = gcs_[gcs_['theta_index'] < theta_cut]['coord_SCs_z']

fig, axes = plt.subplots(2, 3, figsize=(5, 4), width_ratios=[1, 0.5, 0.2], height_ratios=[0.2, 1], gridspec_kw={'wspace': 0.15, 'hspace': 0.15}, sharex='col', sharey='row')

# Top left: ML histogram
axes[0, 0].hist(ml_ngs_nontheta, bins=10, color='#7C7171', alpha=0.7, label='NGS non-theta', density=True)
axes[0, 0].hist(ml_ngs_theta, bins=10, color='#F31010', alpha=0.7, label='NGS theta', density=True)
axes[0, 0].hist(ml_gcs_nontheta, bins=10, color='#7C7171', alpha=0.4, label='GCS non-theta', density=True)
axes[0, 0].hist(ml_gcs_theta, bins=10, color='#F31010', alpha=0.4, label='GCS theta', density=True)
axes[0, 0].axis('off')

# Top middle: AP histogram
axes[0, 1].hist(ap_ngs_nontheta, bins=5, color='#7C7171', alpha=0.7, label='NGS non-theta', density=True)
axes[0, 1].hist(ap_ngs_theta, bins=5, color='#F31010', alpha=0.7, label='NGS theta', density=True)
axes[0, 1].hist(ap_gcs_nontheta, bins=5, color='#7C7171', alpha=0.4, label='GCS non-theta', density=True)
axes[0, 1].hist(ap_gcs_theta, bins=5, color='#F31010', alpha=0.4, label='GCS theta', density=True)
axes[0, 1].axis('off')

# Top right: DV histogram (horizontal)
axes[1, 2].hist(dv_ngs_nontheta, bins=10, color='#7C7171', alpha=0.7, orientation='horizontal', label='NGS non-theta', density=True)
axes[1, 2].hist(dv_ngs_theta, bins=10, color='#F31010', alpha=0.7, orientation='horizontal', label='NGS theta', density=True)
axes[1, 2].hist(dv_gcs_nontheta, bins=10, color='#7C7171', alpha=0.4, orientation='horizontal', label='GCS non-theta', density=True)
axes[1, 2].hist(dv_gcs_theta, bins=10, color='#F31010', alpha=0.4, orientation='horizontal', label='GCS theta', density=True)
axes[1, 2].axis('off')

# Bottom left: ML vs DV scatter with annotation overlays for AP slices
ax_ml_dv = axes[1, 0]
for ap_idx, ap_val in zip(ap_indices, ap_slices):
    ap_colors_i = annotation_colors[ap_idx, :, :]
    unique_colors = np.unique(ap_colors_i)
    for color in unique_colors:
        points = extract_border(ap_colors_i, color, only_border=False)
        ml_points = ml_axis[points[:, 1]]
        dv_points = dv_axis[points[:, 0]]
        ax_ml_dv.scatter(ml_points, dv_points, color=color, s=1, alpha=0.05, zorder=1, rasterized=True)
# NGS
ax_ml_dv.scatter(ml_ngs_nontheta, dv_ngs_nontheta, color='#7C7171', alpha=0.3, s=10, label='NGS non-theta', zorder=2)
ax_ml_dv.scatter(ml_ngs_theta, dv_ngs_theta, color='#F31010', alpha=0.3, s=10, label='NGS theta', zorder=2)
# GCS
ax_ml_dv.scatter(ml_gcs_nontheta, dv_gcs_nontheta, color='#7C7171', alpha=0.15, s=10, label='GCS non-theta', zorder=2)
ax_ml_dv.scatter(ml_gcs_theta, dv_gcs_theta, color='#F31010', alpha=0.15, s=10, label='GCS theta', zorder=2)
ax_ml_dv.set_xlim(2500, 4000)
ax_ml_dv.set_ylim(1000, 5000)
ax_ml_dv.invert_yaxis()
ax_ml_dv.set_xlabel('Medial-lateral (µm)')
ax_ml_dv.set_ylabel('Dorsal-ventral (µm)')

# Bottom middle: AP vs DV scatter (no annotation overlay)
ax_ap_dv = axes[1, 1]
axes[1, 1].scatter(ap_ngs_nontheta, dv_ngs_nontheta, color='#7C7171', alpha=0.15, s=10, label='NGS non-theta')
axes[1, 1].scatter(ap_ngs_theta, dv_ngs_theta, color='#F31010', alpha=0.15, s=10, label='NGS theta')
axes[1, 1].scatter(ap_gcs_nontheta, dv_gcs_nontheta, color='#7C7171', alpha=0.07, s=10, label='GCS non-theta')
axes[1, 1].scatter(ap_gcs_theta, dv_gcs_theta, color='#F31010', alpha=0.07, s=10, label='GCS theta')
axes[1, 1].set_xlim(3500, 5000)
axes[1, 1].set_xlabel('Anterior-posterior (µm)')

# Bottom right: blank
axes[0, 2].axis('off')

plt.tight_layout()
plt.savefig('/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_1_registration/cluster_locations_theta_vs_nontheta.pdf', bbox_inches='tight', dpi=300)
plt.show()